# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
'''

## 1. Unit of analysis + time window

One row represents the daily search performance of a single content item, for one client,
on one calendar day, from `fact_content_daily_performance`.

This notebook uses **March 2026 (`month = '2026-03'`)**, a mid-panel month. The `_sample`
table contains only June 2026 (the final month) and is used only to test query mechanics —
never to build labels, since the final month is the natural outcome window of any
past→future label and would leak the future into the data.

'''

"\n\n## 1. Unit of analysis + time window\n\nOne row represents the daily search performance of a single content item, for one client,\non one calendar day, from `fact_content_daily_performance`.\n\nThis notebook uses **March 2026 (`month = '2026-03'`)**, a mid-panel month. The `_sample`\ntable contains only June 2026 (the final month) and is used only to test query mechanics —\nnever to build labels, since the final month is the natural outcome window of any\npast→future label and would leak the future into the data.\n\n"

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
'''

## 2. Fields: feature / label / context / excluded

### Features (knowable before the refresh decision)
- `impressions` — daily search impressions, summed over the window, before the decision point
- `clicks` — daily search clicks, summed over the window, before the decision point
- `ctr` — computed from clicks/impressions, before the decision point
- `avg_position` — average Search Console ranking position, before the decision point
- `content_age_days` — static content metadata, known at any point in time

### Label
- `is_declining` — constructed by me: comparing impressions in the second half of March
  vs. the first half of March. 1 if impressions declined, 0 otherwise. Built the same way
  the CSV teaching slice's `trend_direction == "down"` label was built, just on a smaller
  window since I only have March loaded.

### Context (for joins/grouping, not features)
- `content_id`, `client_id`, `report_date`, `month`

### Excluded
- Any raw trend/direction column, if present — would directly define the label, causing leakage.
- `provider_used`, `model_used` — describe how content was generated, not its search performance.
- GA4 columns where `ga4_data_available` is FALSE — zero-filled placeholders, not real zeros.

'''

'\n\n## 2. Fields: feature / label / context / excluded\n\n### Features (knowable before the refresh decision)\n- `impressions` — daily search impressions, summed over the window, before the decision point\n- `clicks` — daily search clicks, summed over the window, before the decision point\n- `ctr` — computed from clicks/impressions, before the decision point\n- `avg_position` — average Search Console ranking position, before the decision point\n- `content_age_days` — static content metadata, known at any point in time\n\n### Label\n- `is_declining` — constructed by me: comparing impressions in the second half of March\n  vs. the first half of March. 1 if impressions declined, 0 otherwise. Built the same way\n  the CSV teaching slice\'s `trend_direction == "down"` label was built, just on a smaller\n  window since I only have March loaded.\n\n### Context (for joins/grouping, not features)\n- `content_id`, `client_id`, `report_date`, `month`\n\n### Excluded\n- Any raw trend/direction co

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
'''

## 3. Verification

The following queries verify the data contract:
- the data grain (one row per content item per client per day)
- row count and date range
- missing values
- data availability

'''

'\n\n## 3. Verification\n\nThe following queries verify the data contract:\n- the data grain (one row per content item per client per day)\n- row count and date range\n- missing values\n- data availability\n\n'

In [4]:
'''

### Five features — "available when?"

- **impressions_march** — knowable before the decision because it's a sum of past daily impressions.
- **clicks_march** — knowable before the decision because it's a sum of past daily clicks.
- **ctr** — knowable before the decision because it's calculated purely from past clicks and impressions.
- **avg_position** — knowable before the decision because it's an average of past ranking positions.
- **content_age_days** — knowable at any point because it's static content metadata.

'''

'\n\n### Five features — "available when?"\n\n- **impressions_march** — knowable before the decision because it\'s a sum of past daily impressions.\n- **clicks_march** — knowable before the decision because it\'s a sum of past daily clicks.\n- **ctr** — knowable before the decision because it\'s calculated purely from past clicks and impressions.\n- **avg_position** — knowable before the decision because it\'s an average of past ranking positions.\n- **content_age_days** — knowable at any point because it\'s static content metadata.\n\n'

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [5]:
'''

## 4. Data limits

- The warehouse is an unbalanced panel — different clients have different amounts of
  historical data, so a raw row count isn't directly comparable across clients.
- Some early records have Google Search Console data but not yet Google Analytics data
  (`ga4_data_available = FALSE`) — those rows are zero-filled, not truly zero-engagement.
- Feature and label windows must be aligned carefully. Here, the label is built only from
  the second half of March, so features summed over the full month technically overlap the
  label window slightly — a stricter version would compute features only from the first half.
- External factors (algorithm updates, competitor activity, campaigns) aren't captured here.
- These results are decision-support and directional, not definitive predictions.

'''

"\n\n## 4. Data limits\n\n- The warehouse is an unbalanced panel — different clients have different amounts of\n  historical data, so a raw row count isn't directly comparable across clients.\n- Some early records have Google Search Console data but not yet Google Analytics data\n  (`ga4_data_available = FALSE`) — those rows are zero-filled, not truly zero-engagement.\n- Feature and label windows must be aligned carefully. Here, the label is built only from\n  the second half of March, so features summed over the full month technically overlap the\n  label window slightly — a stricter version would compute features only from the first half.\n- External factors (algorithm updates, competitor activity, campaigns) aren't captured here.\n- These results are decision-support and directional, not definitive predictions.\n\n"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.